<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# 模块 3.5: 面向对象编程
**上一节: [函数式编程](3.4_functional_programming.ipynb)**<br>
**下一节: [类型](3.6_types.ipynb)**

## 动机
Scala 和 Chisel 是面向对象的编程语言，意味着代码可以被划分为对象。
Scala 基于 Java 构建，继承了 Java 的许多面向对象特性。
然而，正如我们将在下面看到的，存在一些差异。
Chisel 的硬件模块类似于 Verilog 的模块，因为它们可以作为单个或多个实例进行实例化和连接。

## 设置

In [ ]:
val path = System.getProperty("user.dir") + "/source/load-ivy.sc"
interp.load.module(ammonite.ops.Path(java.nio.file.FileSystems.getDefault().getPath(path)))

In [ ]:
import chisel3._
import chisel3.util._
import chisel3.experimental._
import chisel3.tester._
import chisel3.tester.RawTester.test

---
# 对象 Oriented Programming
本节概述了 Scala 如何实现面向对象的编程范式。到目前为止，你已经看到了类，但 Scala 还有以下特性：
- [Abstract classes](#abstract)
- [Traits](#traits)
- [Objects](#objects)
- [Companion Objects](#compobj)
- [Case Classes](#caseclass)

## Abstract Classes<a name="abstract"></a>
抽象类就像其他编程语言的实现一样。它们可以定义许多未实现的值，子类必须实现这些值。任何对象只能直接从一个父抽象类继承。

<span style="color:blue">**示例: Abstract 类**</span><br>

In [ ]:
abstract class MyAbstractClass {
  def myFunction(i: Int): Int
  val myValue: String
}
class ConcreteClass extends MyAbstractClass {
  def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}
// 取消注释以下内容来测试！
// val abstractClass = new MyAbstractClass() // 非法！无法实例化抽象类
val concreteClass = new ConcreteClass()      // Legal!


## Traits<a name="traits"></a>
特质与抽象类非常相似，因为它们可以定义未实现的值。然而，它们在两个方面有所不同：
- 一个类可以从多个特质继承
- 特质不能有构造函数参数

<span style="color:blue">**示例: 特质和多重继承**</span><br>
特质是 Scala 实现多重继承的方式，如下面的示例所示。`MyClass` 同时扩展了特质 `HasFunction` 和 `HasValue`：

In [ ]:
trait HasFunction {
  def myFunction(i: Int): Int
}
trait HasValue {
  val myValue: String
  val myOtherValue = 100
}
class MyClass extends HasFunction with HasValue {
  override def myFunction(i: Int): Int = i + 1
  val myValue = "Hello World!"
}
// 取消注释以下内容来测试！
// val myTraitFunction = new HasFunction() // 非法！无法实例化特质
// val myTraitValue = new HasValue()       // 非法！无法实例化特质
val myClass = new MyClass()                // Legal!

要继承多个特质，可以这样链接它们：

```scala
class MyClass extends HasTrait1 with HasTrait2 with HasTrait3 ...
```
一般来说，总是使用特质而不是抽象类，除非您确定要强制执行抽象类的单继承限制。

## Objects<a name="objects"></a>
Scala 有一个用于这些单例类的语言特性，称为对象。您无法实例化对象**(无需调用 `new`)**；您可以直接引用它。这使得它们类似于 Java 静态类。

<span style="color:blue">**示例: Objects**</span><br>

In [ ]:
object MyObject {
  def hi: String = "Hello World!"
  def apply(msg: String) = msg
}
println(MyObject.hi)
println(MyObject("This message is important!")) // 等同于 MyObject.apply(msg)

## Companion Objects<a name="compobj"></a>

当一个类和一个对象共享相同的名称并在同一个文件中定义时，该对象被称为**伴生对象**。当您在类/对象名称之前使用 `new` 时，它将实例化该类。如果您不使用 `new`，它将引用该对象：

<span style="color:blue">**示例: Companion 对象**</span><br>

In [ ]:
object Lion {
    def roar(): Unit = println("I'M AN OBJECT!")
}
class Lion {
    def roar(): Unit = println("I'M A CLASS!")
}
new Lion().roar()
Lion.roar()

Companion objects are usually used for 以下 reasons:
  1. to contain constants related to the 类
  2. to execute code before/after the 类 constructor
  3. to create multiple constructors for a 类

在下面的示例中，我们将实例化多个 Animal 实例。我们希望每个动物都有一个名称，并知道它在所有实例中的顺序。最后，如果没有提供名称，它应该获得一个默认名称。

In [ ]:
object Animal {
    val defaultName = "Bigfoot"
    private var numberOfAnimals = 0
    def apply(name: String): Animal = {
        numberOfAnimals += 1
        new Animal(name, numberOfAnimals)
    }
    def apply(): Animal = apply(defaultName)
}
class Animal(name: String, order: Int) {
  def info: String = s"Hi my name is $name, and I'm $order in line!"
}

val bunny = Animal.apply("Hopper") // Calls the Animal factory method
println(bunny.info)
val cat = Animal("Whiskers")       // Calls the Animal factory method
println(cat.info)
val yeti = Animal()                // Calls the Animal factory method
println(yeti.info)


*这里发生了什么？*
1. 我们的 **Animal 伴生对象** 定义了与 ```class Animal``` 相关的常量：
```scala
val defaultName = "Bigfoot"
```
1. 它还定义了一个私有可变整数来跟踪 Animal 实例的顺序：
```scala 
private var numberOfAnimals = 0
```
1. 它定义了两个 **apply** 方法，这些方法被称为**工厂方法**，因为它们返回 **class Animal** 的实例。
    1. 第一个使用只有一个参数 ```name``` 创建 Animal 实例，并且还使用 ```numberOfAnimals``` 来调用 Animal 类构造函数。
```scala
def apply(name: String): Animal = {
            numberOfAnimals += 1
            new Animal(name, numberOfAnimals)
}
```
    1. 第二个工厂方法不需要参数，而是使用默认名称来调用另一个 apply 方法。
```scala
def apply(): Animal = apply(defaultName)
```
1. These factory methods can be called naively like this
```scala
val bunny = Animal.apply("Hopper")
```
which eliminates the need to use the new keyword, but the real magic is that the compiler assumes the apply 方法 any time it sees parentheses applied to an 实例 or 对象:
```scala
val cat = Animal("Whiskers")
```
1. Factory methods, usually provided via companion objects, allow alternative ways to express 实例 creations, provide additional tests for constructor parameters, conversions, and eliminate the need to use the keyword ```new```. 请注意 you must call the companion 对象's `apply` 方法 for `numberOfAnimals` to be incremented.

**Chisel uses many companion objects, like 模块.** When you write 以下:
```scala
val myModule = 模块(new MyModule)
```
you are calling the **模块 companion 对象**, so Chisel can run background code before and after instantiating 
```MyModule```.

## Case Classes<a name="caseclass"/>
Case classes are a special 类型 of Scala 类 that provides some cool additional features. They are very common in Scala programming, so this section outlines some of their useful features:
- Allows **external access** to the **类 parameters**
- **Eliminates** the need to use **`new`** when instantiating the 类
- Automatically creates an **unapply 方法** that supplies access to all of the 类 Parameters.
- Cannot be subclassed from

In 以下 示例, we declare three different classes, `Nail`, `Screw`, and `Staple`.

In [ ]:
class Nail(length: Int) // Regular class
val nail = new Nail(10) // Requires the `new` keyword
// println(nail.length) // Illegal! Class constructor parameters are not by default externally visible

class Screw(val threadSpace: Int) // By using the `val` keyword, threadSpace is now externally visible
val screw = new Screw(2)          // Requires the `new` keyword
println(screw.threadSpace)

case class Staple(isClosed: Boolean) // Case class constructor parameters are, by default, externally visible
val staple = Staple(false)           // No `new` keyword required
println(staple.isClosed)

`Nail` is a regular 类, and its parameters are not externally visible because we did not use the `val` keyword in the 参数 list. It also requires the `new` keyword when declaring an 实例 of `Nail`.

`Screw` is declared similarly to `Nail`, but includes `val` in the 参数 list. This allows its 参数, `threadSpace`, to be visible externally.

By using a case 类, `Staple` gets the benefit of all its parameters being externally visible (without needing the `val` keyword).

此外, `Staple` does not require using `new` when declaring a case 类. This is because the Scala compiler automatically creates a companion 对象 for every case 类 in your code, which contains an apply 方法 for the case 类.

Case classes are nice containers for generators with lots of parameters.
The constructor gives you a good place to define derived parameters and validate 输入.

In [ ]:
case class SomeGeneratorParameters(
    someWidth: Int,
    someOtherWidth: Int = 10,
    pipelineMe: Boolean = false
) {
    require(someWidth >= 0)
    require(someOtherWidth >= 0)
    val totalWidth = someWidth + someOtherWidth
}

---
# Inheritance with Chisel<a name="super"></a>
You've seen `模块`s and `束`s before, but it's important to realize what's really going on.
Every Chisel 模块 you make is a 类 extending the base 类型 `模块`.
Every Chisel IO you make is a 类 extending the base 类型 `束` (or, in some special cases, `束`'s supertype [`Record`](https://github.com/freechipsproject/chisel3/blob/v3.0.0/chiselFrontend/src/main/scala/chisel3/core/Aggregate.scala#L415)).
Chisel 硬件 types like `UInt` or `束` all have `Data` as a supertype.
We'll explore using 对象 oriented programming to create hierarchical 硬件 blocks and explore 对象 reuse. You'll learn more about types and `Data` in the next 模块 on 类型 generic generators.

## 模块<a name="模块"></a>
Whenever you want to create a 硬件 对象 in Chisel, it needs to have `模块` as a superclass.
Inheritance might not always be the right tool for reuse ([composition over inheritance](https://en.wikipedia.org/wiki/Composition_over_inheritance) is a common principle), but inheritance is still a powerful tool.
下面是 an 示例 of creating a `模块` and connecting multiple instantiations of them together hierarchically.

<span style="color:blue">**示例: Gray Encoder and Decoder**</span><br>
We'll create a 硬件 Gray encoder/decoder. The encode or decode operation choice is 硬件 programmable.

In [ ]:
import scala.math.pow

// create a module
class GrayCoder(bitwidth: Int) extends Module {
  val io = IO(new Bundle{
    val in = Input(UInt(bitwidth.W))
    val out = Output(UInt(bitwidth.W))
    val encode = Input(Bool()) // decode on false
  })
  
  when (io.encode) { //encode
    io.out := io.in ^ (io.in >> 1.U)
  } .otherwise { // decode, much more complicated
    io.out := Seq.fill(log2Ceil(bitwidth))(Wire(UInt(bitwidth.W))).zipWithIndex.fold((io.in, 0)){
      case ((w1: UInt, i1: Int), (w2: UInt, i2: Int)) => {
        w2 := w1 ^ (w1 >> pow(2, log2Ceil(bitwidth)-i2-1).toInt.U)
        (w2, i1)
      }
    }._1
  }
}


Give it a whirl!

In [ ]:
// test our gray coder
val bitwidth = 4
test(new GrayCoder(bitwidth)) { c =>
    def toBinary(i: Int, digits: Int = 8) = {
        String.format("%" + digits + "s", i.toBinaryString).replace(' ', '0')
    }
    println("Encoding:")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(true.B)
        c.clock.step(1)
        println(s"In = ${toBinary(i, bitwidth)}, Out = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

    println("Decoding:")
    for (i <- 0 until pow(2, bitwidth).toInt) {
        c.io.in.poke(i.U)
        c.io.encode.poke(false.B)
        c.clock.step(1)
        println(s"In = ${toBinary(i, bitwidth)}, Out = ${toBinary(c.io.out.peek().litValue.toInt, bitwidth)}")
    }

}

Gray codes are often used in asynchronous interfaces. Usually Gray counters are used rather than fully-featured encoders/decoders, but we'll use the above 模块 to simplify things. 下面是 an 示例 AsyncFIFO, built using the above Gray coder. The control logic and 测试器 is left as an 练习 for later on. For now, look at how the Gray coder is instantiated multiple times and connected.

In [ ]:
class AsyncFIFO(depth: Int = 16) extends Module {
  val io = IO(new Bundle{
    // write inputs
    val write_clock = Input(Clock())
    val write_enable = Input(Bool())
    val write_data = Input(UInt(32.W))
    
    // read inputs/outputs
    val read_clock = Input(Clock())
    val read_enable = Input(Bool())
    val read_data = Output(UInt(32.W))
    
    // FIFO status
    val full = Output(Bool())
    val empty = Output(Bool())
  })
  
  // add extra bit to counter to check for fully/empty status
  assert(isPow2(depth), "AsyncFIFO needs a power-of-two depth!")
  val write_counter = withClock(io.write_clock) { Counter(io.write_enable && !io.full, depth*2)._1 }
  val read_counter = withClock(io.read_clock) { Counter(io.read_enable && !io.empty, depth*2)._1 }
  
  // encode
  val encoder = new GrayCoder(write_counter.getWidth)
  encoder.io.in := write_counter
  encoder.io.encode := true.B
  
  // synchronize
  val sync = withClock(io.read_clock) { ShiftRegister(encoder.io.out, 2) }
  
  // decode
  val decoder = new GrayCoder(read_counter.getWidth)
  decoder.io.in := sync
  decoder.io.encode := false.B
  
  // status logic goes here
  
}

---
# You're done!

[Return to the top.](#top)